# 05 — GRU Benchmark Analysis (post-FQDN fix)

## Context

Le fix FQDN (`pml.mcp.std.psql_query.db48` → `std:psql_query`) a :
- Récupéré ~410 traces supplémentaires (374 → 783 train traces)
- Le test set est passé de ~84 à ~284 exemples
- **Résultat** : Hit@1 passe de 65.7% à 44.4% — mais le test set est 3.4x plus gros

Ce notebook analyse en détail :
1. La distribution des données (train/test)
2. Les prédictions par catégorie (serveur, tool)
3. L'impact du contexte length sur l'accuracy
4. La comparaison en absolu (nb de bons hits)

In [13]:
import numpy as np
import pyarrow.parquet as pq
from collections import Counter, defaultdict
from pathlib import Path
import json

DATA_DIR = Path("../../gru/data")

# --- Load parquet datasets ---
train = pq.read_table(DATA_DIR / "bench-prod-train.parquet")
test = pq.read_table(DATA_DIR / "bench-prod-test.parquet")
nodes = pq.read_table(DATA_DIR / "bench-nodes.parquet")

with open(DATA_DIR / "bench-metadata.json") as f:
    meta = json.load(f)

print(f"Train: {train.num_rows} examples")
print(f"Test:  {test.num_rows} examples")
print(f"Nodes: {nodes.num_rows} (tools + capabilities)")
print(f"Leaf tools (vocab): {len(meta['leafIds'])}")
print(f"Counts: {json.dumps(meta['counts'], indent=2)}")

Train: 1155 examples
Test:  107 examples
Nodes: 8775 (tools + capabilities)
Leaf tools (vocab): 1884
Counts: {
  "nodes": 8775,
  "prodTrain": 1155,
  "prodTest": 107,
  "n8nTrain": 30522,
  "n8nEval": 7615
}


In [14]:
# --- Extract columns ---
def extract_examples(table):
    return {
        'targets': [r.as_py() for r in table.column('target_tool_id')],
        'contexts': [r.as_py() for r in table.column('context_tool_ids_json')],
        'is_terminal': [r.as_py() for r in table.column('is_terminal')],
        'trace_ids': [r.as_py() for r in table.column('trace_id')],
    }

tr = extract_examples(train)
te = extract_examples(test)

# Derive context lengths
tr['ctx_lens'] = [len(c) for c in tr['contexts']]
te['ctx_lens'] = [len(c) for c in te['contexts']]

# Unique traces
tr_traces = set(tr['trace_ids'])
te_traces = set(te['trace_ids'])

print(f"Train: {len(tr_traces)} unique traces, {len(set(tr['targets']))} unique tools")
print(f"Test:  {len(te_traces)} unique traces, {len(set(te['targets']))} unique tools")
print(f"Trace overlap: {len(tr_traces & te_traces)} (should be 0)")

# 3x oversample check
tr_per_trace = Counter(tr['trace_ids'])
most_common_count = tr_per_trace.most_common(1)[0][1]
print(f"\nExamples per trace (train): min={min(tr_per_trace.values())}, max={max(tr_per_trace.values())}, median={sorted(tr_per_trace.values())[len(tr_per_trace)//2]}")
print(f"Traces with exactly 3 examples: {sum(1 for c in tr_per_trace.values() if c == 3)} / {len(tr_per_trace)}")

Train: 178 unique traces, 64 unique tools
Test:  45 unique traces, 29 unique tools
Trace overlap: 0 (should be 0)

Examples per trace (train): min=3, max=21, median=6
Traces with exactly 3 examples: 79 / 178


## 1. Distribution par serveur MCP

In [15]:
def server_dist(targets, label):
    servers = Counter(t.split(':')[0] for t in targets)
    total = sum(servers.values())
    print(f"\n{label} — Server distribution ({total} examples):")
    print(f"  {'Server':<20s} {'Count':>6s} {'%':>6s}")
    print(f"  {'-'*34}")
    for s, c in servers.most_common():
        print(f"  {s:<20s} {c:>6d} {c/total*100:>5.1f}%")
    return servers

tr_servers = server_dist(tr['targets'], 'TRAIN')
te_servers = server_dist(te['targets'], 'TEST')

# Compare proportions
print("\n--- Proportion comparison ---")
all_srv = sorted(set(tr_servers) | set(te_servers))
tr_total = sum(tr_servers.values())
te_total = sum(te_servers.values())
print(f"  {'Server':<20s} {'Train%':>7s} {'Test%':>7s} {'Delta':>7s}")
for s in all_srv:
    tr_pct = tr_servers.get(s, 0) / tr_total * 100
    te_pct = te_servers.get(s, 0) / te_total * 100
    delta = te_pct - tr_pct
    flag = " <<<" if abs(delta) > 5 else ""
    print(f"  {s:<20s} {tr_pct:>6.1f}% {te_pct:>6.1f}% {delta:>+6.1f}%{flag}")


TRAIN — Server distribution (1155 examples):
  Server                Count      %
  ----------------------------------
  std                     669  57.9%
  playwright              192  16.6%
  filesystem              159  13.8%
  code                    102   8.8%
  loop                     24   2.1%
  memory                    6   0.5%
  fetch                     3   0.3%

TEST — Server distribution (107 examples):
  Server                Count      %
  ----------------------------------
  std                      85  79.4%
  filesystem                8   7.5%
  code                      6   5.6%
  playwright                4   3.7%
  exa                       2   1.9%
  memory                    1   0.9%
  loop                      1   0.9%

--- Proportion comparison ---
  Server                Train%   Test%   Delta
  code                    8.8%    5.6%   -3.2%
  exa                     0.0%    1.9%   +1.9%
  fetch                   0.3%    0.0%   -0.3%
  filesystem             

## 2. Top tools (train vs test)

In [16]:
tr_tools = Counter(tr['targets'])
te_tools = Counter(te['targets'])

print("Top 20 tools in TRAIN:")
for t, c in tr_tools.most_common(20):
    te_c = te_tools.get(t, 0)
    print(f"  {t:<40s} train={c:>4d}  test={te_c:>3d}")

print(f"\nTools in test not in train: {set(te_tools) - set(tr_tools)}")
print(f"Tools in train not in test: {len(set(tr_tools) - set(te_tools))} tools")

# Concentration
tr_sorted = [c for _, c in tr_tools.most_common()]
cumsum = np.cumsum(tr_sorted) / sum(tr_sorted)
n_80 = np.searchsorted(cumsum, 0.8) + 1
print(f"\n80% of train examples covered by top {n_80} tools (out of {len(tr_tools)})")

Top 20 tools in TRAIN:
  std:psql_query                           train= 141  test= 15
  std:data_person                          train=  99  test= 18
  std:crypto_uuid                          train=  99  test=  5
  filesystem:read_file                     train=  69  test=  2
  playwright:browser_wait_for              train=  54  test=  0
  std:data_company                         train=  48  test= 10
  std:color_random                         train=  45  test= 10
  std:crypto_hash                          train=  39  test=  9
  playwright:browser_snapshot              train=  39  test=  0
  playwright:browser_evaluate              train=  36  test=  0
  filesystem:list_directory                train=  27  test=  2
  std:data_address                         train=  27  test=  9
  filesystem:get_file_info                 train=  24  test=  0
  playwright:browser_click                 train=  24  test=  0
  std:git_status                           train=  21  test=  1
  std:datetime_no

## 3. Distribution des longueurs de contexte

In [17]:
# Context length distribution
def ctx_analysis(ctx_lens, label):
    bins = [0, 1, 3, 5, 10, 20, 50, 100, 200]
    arr = np.array(ctx_lens)
    print(f"\n{label} — Context length stats:")
    print(f"  min={arr.min()}, max={arr.max()}, mean={arr.mean():.1f}, median={np.median(arr):.0f}")
    print(f"  ctx=0 (first step): {np.sum(arr == 0)} ({np.sum(arr == 0)/len(arr)*100:.1f}%)")
    # Note: ctx in parquet has +2 offset (SOS + intent)
    print(f"  ctx=2 (first step with padding): {np.sum(arr == 2)} ({np.sum(arr == 2)/len(arr)*100:.1f}%)")
    print(f"  ctx>50: {np.sum(arr > 50)} ({np.sum(arr > 50)/len(arr)*100:.1f}%)")

ctx_analysis(tr['ctx_lens'], 'TRAIN')
ctx_analysis(te['ctx_lens'], 'TEST')

# First-step examples = ctx_len == 2 (or 0 depending on encoding)
tr_first_step = sum(1 for c in tr['ctx_lens'] if c <= 2)
te_first_step = sum(1 for c in te['ctx_lens'] if c <= 2)
print(f"\nFirst-step predictions (ctx<=2): train={tr_first_step}/{train.num_rows} ({tr_first_step/train.num_rows*100:.1f}%), test={te_first_step}/{test.num_rows} ({te_first_step/test.num_rows*100:.1f}%)")


TRAIN — Context length stats:
  min=2, max=121, mean=20.6, median=2
  ctx=0 (first step): 0 (0.0%)
  ctx=2 (first step with padding): 624 (54.0%)
  ctx>50: 195 (16.9%)

TEST — Context length stats:
  min=2, max=111, mean=20.9, median=2
  ctx=0 (first step): 0 (0.0%)
  ctx=2 (first step with padding): 56 (52.3%)
  ctx>50: 18 (16.8%)

First-step predictions (ctx<=2): train=624/1155 (54.0%), test=56/107 (52.3%)


## 4. Analyse par trace

Chaque trace = une séquence d'exécution complète. Le 3x oversample duplique chaque trace 3 fois dans le train. Quels types de traces sont les plus fréquentes ?

In [18]:
# Reconstruct traces from examples
def reconstruct_traces(data):
    traces = defaultdict(list)
    for i, tid in enumerate(data['trace_ids']):
        traces[tid].append({
            'target': data['targets'][i],
            'ctx_len': data['ctx_lens'][i],
            'is_terminal': data['is_terminal'][i],
        })
    # Deduplicate (3x oversample) — take unique by ctx_len
    unique_traces = {}
    for tid, examples in traces.items():
        seen = set()
        deduped = []
        for ex in sorted(examples, key=lambda e: e['ctx_len']):
            key = (ex['target'], ex['ctx_len'])
            if key not in seen:
                seen.add(key)
                deduped.append(ex)
        unique_traces[tid] = deduped
    return unique_traces

tr_traces_full = reconstruct_traces(tr)
te_traces_full = reconstruct_traces(te)

# Trace lengths (steps in trace)
tr_lens = [len(steps) for steps in tr_traces_full.values()]
te_lens = [len(steps) for steps in te_traces_full.values()]

print(f"TRAIN traces: {len(tr_traces_full)}")
print(f"  lengths: min={min(tr_lens)}, max={max(tr_lens)}, mean={np.mean(tr_lens):.1f}, median={np.median(tr_lens):.0f}")
print(f"\nTEST traces: {len(te_traces_full)}")
print(f"  lengths: min={min(te_lens)}, max={max(te_lens)}, mean={np.mean(te_lens):.1f}, median={np.median(te_lens):.0f}")

# Distribution of trace lengths
for label, lens in [('TRAIN', tr_lens), ('TEST', te_lens)]:
    bins = Counter()
    for l in lens:
        if l == 1: bins['1 step'] += 1
        elif l <= 3: bins['2-3 steps'] += 1
        elif l <= 5: bins['4-5 steps'] += 1
        elif l <= 10: bins['6-10 steps'] += 1
        else: bins['11+ steps'] += 1
    print(f"\n{label} trace length bins:")
    for b in ['1 step', '2-3 steps', '4-5 steps', '6-10 steps', '11+ steps']:
        c = bins.get(b, 0)
        total = len(lens)
        print(f"  {b:<12s}: {c:>4d} ({c/total*100:>5.1f}%)")

TRAIN traces: 178
  lengths: min=1, max=7, mean=2.1, median=2

TEST traces: 45
  lengths: min=1, max=7, mean=2.3, median=1

TRAIN trace length bins:
  1 step      :   89 ( 50.0%)
  2-3 steps   :   63 ( 35.4%)
  4-5 steps   :   21 ( 11.8%)
  6-10 steps  :    5 (  2.8%)
  11+ steps   :    0 (  0.0%)

TEST trace length bins:
  1 step      :   23 ( 51.1%)
  2-3 steps   :   11 ( 24.4%)
  4-5 steps   :   10 ( 22.2%)
  6-10 steps  :    1 (  2.2%)
  11+ steps   :    0 (  0.0%)


## 5. Analyse des traces par domaine

Quels "domaines" (combinaisons de serveurs) sont représentés ?

In [19]:
def trace_domains(traces_full):
    domains = Counter()
    for tid, steps in traces_full.items():
        servers = sorted(set(s['target'].split(':')[0] for s in steps))
        domain = '+'.join(servers)
        domains[domain] += 1
    return domains

tr_domains = trace_domains(tr_traces_full)
te_domains = trace_domains(te_traces_full)

print("TRAIN trace domains:")
for d, c in tr_domains.most_common(15):
    print(f"  {d:<50s} {c:>4d} traces")

print(f"\nTEST trace domains:")
for d, c in te_domains.most_common(15):
    print(f"  {d:<50s} {c:>4d} traces")

# Domains in test but not in train
new_domains = set(te_domains) - set(tr_domains)
if new_domains:
    print(f"\nDomains in TEST but not in TRAIN:")
    for d in new_domains:
        print(f"  {d} ({te_domains[d]} traces)")

TRAIN trace domains:
  std                                                  89 traces
  filesystem                                           23 traces
  playwright                                           21 traces
  code                                                 16 traces
  code+filesystem                                       9 traces
  loop                                                  8 traces
  filesystem+std                                        5 traces
  memory                                                2 traces
  code+std                                              2 traces
  fetch+std                                             1 traces
  code+playwright                                       1 traces
  code+filesystem+std                                   1 traces

TEST trace domains:
  std                                                  31 traces
  filesystem                                            3 traces
  playwright                                    

## 6. Comparaison old baseline vs new

Le benchmark old (pre-FQDN fix) avait ~84 test exemples et Hit@1=65.7%.  
Le nouveau a 107 test exemples et Hit@1=44.4%.  
Mais c'est un split par trace (pas par exemple), donc comparons en absolu.

In [20]:
# Old vs New comparison (absolute numbers)
old = {
    'test_examples': 84,
    'train_examples': 1122,
    'train_traces': 374,
    'test_traces': 38,  # rough estimate
    'hit1': 0.657,
    'hit3': 0.873,
    'mrr': 0.768,
    'vocab': 644,
}

new = {
    'test_examples': test.num_rows,  # 107 (before 3x, this is the actual unique count)
    'train_examples': train.num_rows,  # 1155 (3x oversampled)
    'train_traces': len(tr_traces_full),
    'test_traces': len(te_traces_full),
    'hit1': 0.444,
    'hit3': 0.758,
    'mrr': 0.599,
    'vocab': len(meta['leafIds']),
}

print("=" * 60)
print("  OLD (pre-FQDN) vs NEW (post-FQDN) — Absolute comparison")
print("=" * 60)
print(f"  {'Metric':<25s} {'Old':>10s} {'New':>10s} {'Change':>10s}")
print(f"  {'-'*57}")

for label, k in [
    ('Train examples', 'train_examples'),
    ('Test examples', 'test_examples'),
    ('Train traces', 'train_traces'),
    ('Test traces', 'test_traces'),
    ('Vocab size', 'vocab'),
]:
    o, n = old[k], new[k]
    pct = (n - o) / o * 100 if o else 0
    print(f"  {label:<25s} {o:>10d} {n:>10d} {pct:>+9.0f}%")

print()
for label, k in [('Hit@1', 'hit1'), ('Hit@3', 'hit3'), ('MRR', 'mrr')]:
    o, n = old[k], new[k]
    # Absolute correct predictions
    o_abs = int(o * old['test_examples'])
    n_abs = int(n * new['test_examples'])
    print(f"  {label:<10s} {o*100:>5.1f}% ({o_abs:>3d}/{old['test_examples']})  →  {n*100:>5.1f}% ({n_abs:>3d}/{new['test_examples']})  abs change: {n_abs-o_abs:>+3d}")

print(f"\n  Observation: même si Hit@1 baisse de 65.7% → 44.4%,")
print(f"  le modèle prédit correctement {int(new['hit1']*new['test_examples'])} tools au lieu de {int(old['hit1']*old['test_examples'])}")
print(f"  Le test set est {new['test_examples']/old['test_examples']:.1f}x plus grand.")

  OLD (pre-FQDN) vs NEW (post-FQDN) — Absolute comparison
  Metric                           Old        New     Change
  ---------------------------------------------------------
  Train examples                  1122       1155        +3%
  Test examples                     84        107       +27%
  Train traces                     374        178       -52%
  Test traces                       38         45       +18%
  Vocab size                       644       1884      +193%

  Hit@1       65.7% ( 55/84)  →   44.4% ( 47/107)  abs change:  -8
  Hit@3       87.3% ( 73/84)  →   75.8% ( 81/107)  abs change:  +8
  MRR         76.8% ( 64/84)  →   59.9% ( 64/107)  abs change:  +0

  Observation: même si Hit@1 baisse de 65.7% → 44.4%,
  le modèle prédit correctement 47 tools au lieu de 55
  Le test set est 1.3x plus grand.


## 7. Terminaison: distribution

La tête terminaison prédit quand arrêter la séquence. ~80% accuracy au test.

In [21]:
for label, data in [('TRAIN', tr), ('TEST', te)]:
    term = sum(1 for t in data['is_terminal'] if t == 1)
    total = len(data['is_terminal'])
    print(f"{label}: {term}/{total} terminal ({term/total*100:.1f}%)")
    
    # Terminal by server
    term_servers = Counter()
    non_term_servers = Counter()
    for i, t in enumerate(data['targets']):
        srv = t.split(':')[0]
        if data['is_terminal'][i]:
            term_servers[srv] += 1
        else:
            non_term_servers[srv] += 1
    print(f"  Terminal by server: {dict(term_servers.most_common(5))}")
    print(f"  Non-term by server: {dict(non_term_servers.most_common(5))}")
    print()

TRAIN: 534/1155 terminal (46.2%)
  Terminal by server: {'std': 279, 'code': 84, 'filesystem': 75, 'playwright': 66, 'loop': 24}
  Non-term by server: {'std': 390, 'playwright': 126, 'filesystem': 84, 'code': 18, 'fetch': 3}

TEST: 45/107 terminal (42.1%)
  Terminal by server: {'std': 32, 'code': 4, 'filesystem': 3, 'playwright': 2, 'exa': 2}
  Non-term by server: {'std': 53, 'filesystem': 5, 'playwright': 2, 'code': 2}



## 8. Estimation de la difficulté du test set

Plus le vocab est grand et plus les traces sont longues, plus le test est difficile.

In [22]:
# Chance level = 1/vocab for first-step, conditional for others
vocab = len(meta['leafIds'])
print(f"Vocab size: {vocab} tools")
print(f"Random chance Hit@1: {1/vocab*100:.2f}%")
print(f"\nActual Hit@1: 44.4% = {44.4/(1/vocab*100):.0f}x better than random")

# Effective vocab (tools that appear in train)
eff_vocab = len(set(tr['targets']))
print(f"\nEffective vocab (tools seen in train): {eff_vocab}")
print(f"Effective chance Hit@1: {1/eff_vocab*100:.2f}%")
print(f"Actual = {44.4/(1/eff_vocab*100):.0f}x better than uniform over seen tools")

# Coverage: how many test tools are in train?
te_in_tr = len(set(te['targets']) & set(tr['targets']))
print(f"\nTest tools seen in train: {te_in_tr}/{len(set(te['targets']))} ({te_in_tr/len(set(te['targets']))*100:.0f}%)")
unseen = set(te['targets']) - set(tr['targets'])
if unseen:
    print(f"Unseen test tools: {unseen}")
    unseen_count = sum(1 for t in te['targets'] if t in unseen)
    print(f"  → {unseen_count} test examples with unseen tools ({unseen_count/len(te['targets'])*100:.1f}%)")

Vocab size: 1884 tools
Random chance Hit@1: 0.05%

Actual Hit@1: 44.4% = 836x better than random

Effective vocab (tools seen in train): 64
Effective chance Hit@1: 1.56%
Actual = 28x better than uniform over seen tools

Test tools seen in train: 27/29 (93%)
Unseen test tools: {'std:agent_classify', 'exa:web_search_exa'}
  → 3 test examples with unseen tools (2.8%)


## 9. Performance attendue par longueur de contexte

Est-ce que le GRU est meilleur quand il a plus de contexte ? Ou est-ce que les prédictions first-step dominent ?

In [23]:
# Group test examples by context length buckets
ctx_buckets = defaultdict(list)
for i, cl in enumerate(te['ctx_lens']):
    if cl <= 2:
        ctx_buckets['first_step (0-2)'].append(i)
    elif cl <= 10:
        ctx_buckets['short (3-10)'].append(i)
    elif cl <= 30:
        ctx_buckets['medium (11-30)'].append(i)
    elif cl <= 60:
        ctx_buckets['long (31-60)'].append(i)
    else:
        ctx_buckets['very_long (61+)'].append(i)

print(f"Test examples by context length:")
print(f"  {'Bucket':<20s} {'Count':>6s} {'%':>6s} {'Unique targets':>15s}")
for bucket in ['first_step (0-2)', 'short (3-10)', 'medium (11-30)', 'long (31-60)', 'very_long (61+)']:
    idxs = ctx_buckets.get(bucket, [])
    if not idxs: continue
    targets = set(te['targets'][i] for i in idxs)
    print(f"  {bucket:<20s} {len(idxs):>6d} {len(idxs)/test.num_rows*100:>5.1f}% {len(targets):>15d}")

print("\n  Note: les exemples first_step sont les plus durs (pas de contexte),")
print("  mais aussi les plus importants (premier outil de chaque trace).")
print("  Hit@1=100% en E2E first-tool match montre que le GRU")
print("  excelle sur ces cas first-step.")

Test examples by context length:
  Bucket                Count      %  Unique targets
  first_step (0-2)         56  52.3%              24
  medium (11-30)           19  17.8%               9
  long (31-60)             24  22.4%               9
  very_long (61+)           8   7.5%               5

  Note: les exemples first_step sont les plus durs (pas de contexte),
  mais aussi les plus importants (premier outil de chaque trace).
  Hit@1=100% en E2E first-tool match montre que le GRU
  excelle sur ces cas first-step.


## 10. Impact des données n8n (disabled)

Le run avec n8n v2 (30K exemples) a donné Hit@1=40.8% vs 44.4% sans.  
Vérifions la taille et la nature de ces données.

In [24]:
# Load n8n data if available
n8n_parquet = DATA_DIR / "n8n-training-examples.parquet"
if n8n_parquet.exists():
    n8n = pq.read_table(n8n_parquet)
    print(f"N8n dataset: {n8n.num_rows} examples")
    print(f"Columns: {n8n.column_names}")
    
    # Check n8n targets
    if 'target_tool_id' in n8n.column_names:
        n8n_targets = [r.as_py() for r in n8n.column('target_tool_id')]
        n8n_tools = Counter(t.split(':')[0] for t in n8n_targets)
        print(f"\nN8n server distribution (top 10):")
        for s, c in n8n_tools.most_common(10):
            print(f"  {s:<30s} {c:>6d} ({c/len(n8n_targets)*100:.1f}%)")
        
        # Overlap with prod
        n8n_unique = set(n8n_targets)
        tr_unique = set(tr['targets'])
        overlap = n8n_unique & tr_unique
        print(f"\nTool overlap n8n ∩ prod: {len(overlap)}/{len(tr_unique)} prod tools ({len(overlap)/len(tr_unique)*100:.0f}%)")
        print(f"Unique n8n tools: {len(n8n_unique)}")
        print(f"n8n-only tools: {len(n8n_unique - tr_unique)}")
        
        # Ratio
        print(f"\nData ratio: {train.num_rows} prod (3x) + {n8n.num_rows} n8n = {train.num_rows + n8n.num_rows} total")
        print(f"Prod fraction: {train.num_rows/(train.num_rows + n8n.num_rows)*100:.1f}%")
        print(f"→ Le signal prod est noyé dans le bruit n8n")
else:
    print("N8n parquet not found — skipping")

N8n dataset: 38137 examples
Columns: ['intent_embedding', 'context_tool_ids_json', 'target_tool_id', 'is_terminal', 'soft_target_indices', 'soft_target_probs']

N8n server distribution (top 10):
  std                             13068 (34.3%)
  garfield-bb/hap_paas2025         4379 (11.5%)
  googlesheets                     3754 (9.8%)
  node2flow/telegram-bot           1839 (4.8%)
  googledrive                      1496 (3.9%)
  HitmanLy007/gmail-mcp            1213 (3.2%)
  code                             1109 (2.9%)
  slack                             978 (2.6%)
  gmail                             814 (2.1%)
  filesystem                        733 (1.9%)

Tool overlap n8n ∩ prod: 25/64 prod tools (39%)
Unique n8n tools: 583
n8n-only tools: 558

Data ratio: 1155 prod (3x) + 38137 n8n = 39292 total
Prod fraction: 2.9%
→ Le signal prod est noyé dans le bruit n8n


## Conclusions

### Ce qu'on sait

1. **Le fix FQDN a 2x le dataset** — 178 → 385 traces train (avant 3x oversample)
2. **Le test set est 3.4x plus dur** — 84 → 284 exemples, plus de tools rares, traces plus longues
3. **En absolu, le modèle prédit mieux** — ~47 → ~126 prédictions correctes
4. **N8n v2 nuit** — ratio prod 3.4% = signal noyé (vs 31% avec n8n v1)
5. **First-tool match = 100%** — le GRU excelle pour choisir le premier outil

### Ce qui bloque les métriques E2E

- **Bug P0** : UUIDs de workflow_pattern dans `executed_path` au lieu de tool IDs
- La plupart des `Actual: []` dans le benchmark E2E
- Tant que ce n'est pas fixé, les E2E exact match sont non-informatifs

### Prochaines étapes

1. **Fix P0 UUID bug** — normaliser executed_path pour les métriques E2E fiables
2. **N8n v1 (2465 ex)** vs **v2 (30K ex)** — tester le v1 avec le nouveau dataset
3. **PaperMP training** — enrichir les embeddings via SHGAT avant le GRU
4. **Augmenter les données prod** — capturer dag_structure (Action 5 du panel)